#DC Task

-Download and import the Data Science Job Salary dataset.

-Normalize the ‘salary’ column using Min-Max normalization which scales all salary values between 0 and 1.

-Implement dimensionality reduction like Principal Component Analysis (PCA) or t-SNE to reduce the number of features (columns) in the dataset.

-Group the dataset by the ‘experience_level’ column and calculate the average and median salary for each experience level (e.g., Junior, Mid-level, Senior).


In [3]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.decomposition import PCA
import numpy as np

# --- Download and import the Data Science Job Salary dataset ---
print("\n--- Importing 'datascience_salaries.csv' dataset ---")
df = pd.read_csv('/content/datascience_salaries.csv')
print("Dataset loaded successfully. Original columns:", df.columns.tolist())
print("First 5 rows:")
print(df.head())

# Drop 'Unnamed: 0' column if it exists, as it often is an artifact from CSV export
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])
    print("Dropped 'Unnamed: 0' column.")

# --- Task 1: Normalize the ‘salary’ column using Min-Max normalization ---
print("\n--- 1. Normalizing 'salary' column using Min-Max normalization ---")
if 'salary' in df.columns:
    scaler = MinMaxScaler()
    df['salary_normalized'] = scaler.fit_transform(df[['salary']])
    print("First 5 rows with original and normalized salary:")
    print(df[['salary', 'salary_normalized']].head())
else:
    print("'salary' column not found in the DataFrame. Skipping normalization.")

# --- Task 2: Implement dimensionality reduction (PCA) ---
print("\n--- 2. Implementing PCA for dimensionality reduction ---")

# Identify numerical and categorical features for PCA
numerical_cols = df.select_dtypes(include=np.number).columns.tolist()
categorical_cols = df.select_dtypes(include='object').columns.tolist()

# Exclude original 'salary' and the newly created 'salary_normalized' from features for PCA
features_to_exclude = ['salary', 'salary_normalized']
numerical_features_for_pca = [col for col in numerical_cols if col not in features_to_exclude]

# One-hot encode categorical features
if categorical_cols:
    print(f"One-hot encoding categorical columns: {categorical_cols}")
    encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    encoded_features = encoder.fit_transform(df[categorical_cols])
    encoded_feature_names = encoder.get_feature_names_out(categorical_cols)
    df_encoded = pd.DataFrame(encoded_features, columns=encoded_feature_names, index=df.index)

    # Combine numerical and encoded categorical features
    df_for_pca = pd.concat([df[numerical_features_for_pca], df_encoded], axis=1)
else:
    df_for_pca = df[numerical_features_for_pca]

# Handle potential NaNs by filling with mean (a simple imputation strategy) for numerical columns
# and mode for encoded categorical columns (which are 0/1)
# For simplicity, we'll fill all NaNs with 0 as one-hot encoded features will be 0/1 and other numerical features can also handle 0 or mean.
# A more robust approach might be to use different imputation strategies per column type.
df_for_pca = df_for_pca.fillna(0)

# Drop columns with zero variance, as PCA cannot work with them
df_for_pca = df_for_pca.loc[:, df_for_pca.var() != 0]

if df_for_pca.shape[1] > 1: # PCA requires at least 2 features to reduce to 2 components
    pca = PCA(n_components=min(2, df_for_pca.shape[1])) # Reduce to 2 components or fewer if less features exist
    principal_components = pca.fit_transform(df_for_pca)
    df_pca_components = pd.DataFrame(data = principal_components, columns = [f'principal_component_{i+1}' for i in range(principal_components.shape[1])])

    print(f"Total features used for PCA: {df_for_pca.shape[1]}")
    print(f"Explained variance ratio by principal components: {pca.explained_variance_ratio_.sum():.2f}")
    print("Individual explained variance ratio:", pca.explained_variance_ratio_)
    print("First 5 rows of PCA components:")
    print(df_pca_components.head())

    # Optionally, you can merge these PCA components back to your original DataFrame:
    # df = pd.concat([df, df_pca_components], axis=1)
else:
    print("Not enough suitable numerical features (more than 1) available for PCA after processing and encoding.")
    print(f"Features considered for PCA: {df_for_pca.columns.tolist()}")

# --- Task 3: Group the dataset by ‘experience_level’ and calculate average and median salary ---
print("\n--- 3. Calculating average and median salary by 'experience_level' ---")
if 'experience_level' in df.columns and 'salary' in df.columns:
    salary_stats = df.groupby('experience_level')['salary'].agg(['mean', 'median'])
    print("Average and Median Salary by Experience Level:")
    print(salary_stats)
else:
    print("Required columns ('experience_level' or 'salary') not found for aggregation. Skipping aggregation.")



--- Importing 'datascience_salaries.csv' dataset ---
Dataset loaded successfully. Original columns: ['Unnamed: 0', 'job_title', 'job_type', 'experience_level', 'location', 'salary_currency', 'salary']
First 5 rows:
   Unnamed: 0       job_title   job_type experience_level       location  \
0           0  Data scientist  Full Time           Senior  New York City   
1           2  Data scientist  Full Time           Senior         Boston   
2           3  Data scientist  Full Time           Senior         London   
3           4  Data scientist  Full Time           Senior         Boston   
4           5  Data scientist  Full Time           Senior  New York City   

  salary_currency  salary  
0             USD  149000  
1             USD  120000  
2             USD   68000  
3             USD  120000  
4             USD  149000  
Dropped 'Unnamed: 0' column.

--- 1. Normalizing 'salary' column using Min-Max normalization ---
First 5 rows with original and normalized salary:
   salary  s